In [21]:
!pip install lightkurve astropy numpy pandas matplotlib 
!pip install scikit-learn torch torchvision 
!pip install transitleastsquares batman-package
!pip install streamlit plotly scipy

In [22]:
!pip install wotan
!pip install transitleastsquares

In [23]:
import numpy as np 
import lightkurve as lk
import matplotlib.pyplot as plt
from wotan import flatten
from transitleastsquares import transitleastsquares

In [24]:
def load_light_curve(target="TIC 307210830", author="SPOC"):
    search=lk.search_lightcurve(target, mission="TESS", author=author)
    print(f"Found {len(search)} light curve files for {target}")

    lc = search[0].download()

    lc=lc.remove_nans().normalize()  #normalize coverts each value into values around 1

    time=lc.time.value
    flux=lc.flux.value

    return time, flux

In [25]:
"""Remove slow stellar variability while keeping the sharp transit dips.
 
    window_length is in DAYS. Keep it several times longer than a transit
    (transits last hours) so the flattening doesn't erase the signal.
"""

def detrend(time, flux, window_length=0.5):
    flat_flux, trend=flatten(
        time, flux,
        window_length=window_length,
        method="biweight",
        return_trend=True,
    )
    good = np.isfinite(flat_flux)
    return time[good], flat_flux[good], trend[good]


In [26]:
def detect(time, flat_flux):
    """Search for repeating transit-shaped dips with TLS."""
    model = transitleastsquares(time, flat_flux)

    results = model.power(period_min=0.5, period_max=10.0)

    dip = 1.0 - results.depth
    print("\n--- Detection ---")
    print(f"Period       : {results.period:.5f} days")
    print(f"Mid-transit  : {results.T0:.4f} (T0)")
    print(f"Duration     : {results.duration * 24:.2f} hours")
    print(f"Transit depth: {dip * 1e6:.0f} ppm ({dip*100:.4f} %)")
    print(f"SDE (signif.): {results.SDE:.1f}")
    print(f"SNR          : {results.snr:.1f}")
    return results

In [27]:
def plot(time, flux, trend, results):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7))

    ax1.plot(time, flux, ".", ms=1, alpha=0.4, label="raw flux")
    ax1.plot(time, trend, "-", lw=1.2, label="wotan trend")
    ax1.set_xlabel("Time (days)")
    ax1.set_ylabel("Normalized flux")
    ax1.set_title("Stage 1-2: raw light curve + fitted trend")
    ax1.legend(loc="lower left")

    ax2.plot(results.folded_phase, results.folded_y, ".", ms=1, alpha=0.4)
    ax2.plot(results.model_folded_phase, results.model_folded_model,
            "-", lw=2, label="TLS model")
    ax2.set_xlabel("Phase")
    ax2.set_ylabel("Normalized flux")
    ax2.set_title(f"Stage 3: phase-folded on P = {results.period:.4f} d")
    ax2.legend(loc="lower left")
    
    fig.tight_layout()
    fig.savefig("detection_result.png", dpi=130)
    print("\nSaved plot -> detection_result.png")

In [28]:
if __name__ == "__main__":
    time, flux = load_light_curve()
    time, flat_flux, trend = detrend(time, flux)
    results = detect(time, flat_flux)
    plot(time, flux, trend, results)

Found 47 light curve files for TIC 307210830
Transit Least Squares TLS 1.32 (5 Apr 2024)
Creating model cache for 36 durations
Searching 17578 data points, 2261 periods from 0.601 to 9.982 days
Using all 16 CPU threads


100%|██████████| 2261/2261 periods | 00:12<00:00


Searching for best T0 for period 3.69144 days


100%|██████████| 7675/7675 [00:02<00:00, 3415.34it/s]



--- Detection ---
Period       : 3.69144 days
Mid-transit  : 1356.2009 (T0)
Duration     : 1.09 hours
Transit depth: 1717 ppm (0.1717 %)
SDE (signif.): 20.9
SNR          : 28.0

Saved plot -> detection_result.png


In [29]:
def extract_features(time, flux, period, t0, duration):

   """Measure the physical fingerprints of a periodic dip.
 
    Returns a dict of features. Each one has a physical meaning:
 
    primary_depth_ppm  : how much light is blocked. Planets are shallow
                         (tiny object); star-on-star eclipses are deep.
    secondary_depth_ppm: is there a SECOND dip half an orbit later? A companion
                         star passing behind produces one; a planet basically
                         does not. The single strongest binary discriminator.
    secondary_ratio    : secondary depth / primary depth. ~0 for planets,
                         clearly positive for eclipsing binaries.
    odd_even_diff      : do alternating transits have different depths? Planets
                         transit the same star every time (=> ~0). Two unequal
                         stars produce alternating deep/shallow eclipses.
    v_shape            : sharpness of the dip bottom. Planets sit fully on the
                         disk (flat, U-shaped bottom); grazing binaries just
                         clip the edge (pointy, V-shaped). Higher = more V-like.
    """
   dur_phase=duration/period 
   half=dur_phase/2.0 

   phase=((time-t0+0.5*period)%period)/period-0.5

   out=np.abs(phase)>3*half 
   baseline=np.median(flux[out]) if out.sum() else 1.0

   in_primary=np.abs(phase) < half 
   primary_depth=baseline-np.median(flux[in_primary]) 
   primary_depth=max(primary_depth, 1e-9) 
   in_secondary=np.abs(np.abs(phase)-0.5) < half
   secondary_depth=baseline-np.median(flux[in_secondary]) if in_secondary.sum() else 0.0

   transit_num = np.round((time - t0) / period).astype(int)
   odd_mask = in_primary & (transit_num % 2==1)
   even_mask = in_primary & (transit_num%2==0)
   d_odd=baseline - np.median(flux[odd_mask]) if odd_mask.sum() else primary_depth
   d_even=baseline - np.median(flux[even_mask]) if even_mask.sum() else primary_depth
   odd_even_diff = abs(d_odd - d_even)/primary_depth

   core=np.abs(phase) < half*0.3
   wing=(np.abs(phase) > half*0.6) & in_primary
   if core.sum() and wing.sum():
    core_depth=baseline-np.median(flux[core])
    wing_depth=baseline-np.median(flux[wing])
    v_shape=1.0-(wing_depth/max(core_depth, 1e-9))
      
   else:
    v_shape=np.nan
   return {
    "primary_depth_ppm": primary_depth*1e6,
    "secondary_depth_ppm": secondary_depth*1e6,
    "secondary_ratio": secondary_depth/primary_depth,
    "odd_even_diff": odd_even_diff,
    "v_shape": v_shape,
   } 

In [30]:
import batman

def inject_transit(time, flux, period, t0, rp, a=15.0, inc=89.5, u=(0.4,0.3)):
    """Multiply a batman transit model into an existing (real) flux array.
 
    time, flux : the real light curve, BEFORE detrending
    period     : orbital period in days
    t0         : time of first transit
    rp         : planet / star radius ratio  (transit depth ~ rp**2, before
                 limb darkening deepens it a little), suppsose planet radius=1 and star rad=10 then rp=1/10=0.1
    a          : orbital distance in stellar radii(or distance from the star)
    inc        : orbital inclination in degrees (90 = perfectly edge-on)
    u          : quadratic limb-darkening coefficients(if the Sun equally bright everywhere? NO....the center is brighter, the edges are darker; this is called limb darkening.)
 
    Returns the flux with the transit imprinted on it.
    """

    p=batman.TransitParams()
    p.t0, p.per, p.rp = t0, period, rp
    p.a, p.inc, p.ecc, p.w = a, inc, 0.0, 90.0 
    p.u, p.limb_dark=list(u), "quadratic" 
    transit_model=batman.TransitModel(p, time).light_curve(p) 
    return flux*transit_model

In [31]:
def random_planet_params(rng, time_span):
    return{
        "period": rng.uniform(1.0, time_span/3.0), 
        "t0": rng.uniform(0.5, 3.0),
        "rp": rng.uniform(0.02, 0.12),
    }

In [32]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

_MEANING = {
    "primary_depth_ppm": "transit depth",
    "secondary_ratio": "presence of a secondary eclipse",
    "odd_even_diff": "difference between alternating transits",
    "v_shape": "V-shaped (grazing) profile",
    "sde": "detection significance",
    "snr": "signal-to-noise", 
}

In [33]:
def train(table, test_size=0.25, random_state=42):
    X=table.drop(columns="label")
    y=table["label"]
    Xtr, Xte, ytr, yte=train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)
    clf=RandomForestClassifier(n_estimators=300, random_state=random_state)
    clf.fit(Xtr, ytr)

    print(classification_report(yte, clf.predict(Xte), digits=2)) 
    print("Confusion matrix(rows=truth, cols=predicted):")
    print("labels:", list(clf.classes_))
    print(confusion_matrix(yte, clf.predict(Xte)))
    return clf, list(X.columns)

In [34]:
def explain(clf, columns, row): 
    """Return a plain-English physical reason for one prediction."""
    import shap
    pred=clf.predict(row)[0] 
    proba=clf.predict_proba(row)[0].max() 
    classes=list(clf.classes_)

    sv=shap.TreeExplainer(clf).shap_values(row)
    idx=classes.index(pred)
    contrib=sv[..., idx][0] if np.ndim(sv)==3 else sv[idx][0]

    ranked=sorted(zip(columns, contrib), key=lambda t: -t[1])
    reasons=[_MEANING.get(f, f) for f, c in ranked[:2] in ranked[:2] if c>0]

    reason_txt=" and ".join(reasons) if reasons else "the overall pattern"
    label=pred.replace("-", " ")
    return f"Classified as {label} (confidence {proba:.%}) mainly due to {reason_txt}."

In [35]:
"""
Two deliverables the problem statement explicitly rewards:
 
1. calibrate()        -> makes reported confidence trustworthy. A raw random
                         forest that says "90%" is often wrong about that number;
                         calibration makes "90% confidence" mean right ~90% of
                         the time.
 
2. completeness_map() -> injects planets across a grid of size x period, runs
                         the REAL pipeline, and measures the recovery fraction.
                         The resulting heatmap is your pipeline's detection
                         floor -- a professional-grade evaluation figure.
"""

import io
import contextlib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt 
from sklearn.calibration import CalibratedClassifierCV

In [36]:
def calibrate(clf, X, y):
    return CalibratedClassifierCV(clf, method="isotonic", cv=5).fit(X,y)

In [37]:
def _recovered(time, rawflux, period, t0, rp, sde_threshold=9.0):
    """Inject one known planet, run YOUR pipeline, return True if recovered."""
    f = inject_transit(time, rawflux, period, t0, rp)  
    time_c, flat, _ = detrend(time, f)                  
    with contextlib.redirect_stdout(io.StringIO()):     
        r = transitleastsquares(time_c, flat).power(period_min=0.5, period_max=10.0)
    return (abs(r.period - period) / period < 0.02) and (r.SDE > sde_threshold)


def completeness_map(time, quiet_flux, radii, periods, trials=5, noise_std=0.0006):
    """Inject a grid of planets into a real quiet star and measure recovery.
    Slow: runs the full pipeline (radii x periods x trials) times.
    Use a coarse grid while testing, a fine one for the final figure."""
    rng = np.random.default_rng(0)
    grid = np.zeros((len(radii), len(periods)))
    for i, rp in enumerate(radii):
        for j, P in enumerate(periods):
            hits = 0
            for _ in range(trials):
                star = quiet_flux + rng.normal(0, noise_std, len(quiet_flux))
                hits += _recovered(time, star, P, rng.uniform(0.5, 2.5), rp)
            grid[i, j] = hits / trials

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(grid, origin="lower", aspect="auto", cmap="viridis", vmin=0, vmax=1)
    ax.set_xticks(range(len(periods)), [f"{p:g}" for p in periods])
    ax.set_yticks(range(len(radii)),  [f"{r:g}" for r in radii])
    ax.set_xlabel("Orbital period (days)")
    ax.set_ylabel("Planet/star radius ratio")
    ax.set_title("Completeness: fraction of injected planets recovered")
    for i in range(len(radii)):
        for j in range(len(periods)):
            ax.text(j, i, f"{grid[i, j]*100:.0f}%", ha="center", va="center",
                    color="white" if grid[i, j] < 0.6 else "black")
    fig.colorbar(im, label="recovery fraction")
    plt.show()
    return grid

In [38]:
import os, io, contextlib
import pandas as pd

def process_one_star(tic, clf=None, feature_cols=None):
    """Run ONE star through the whole pipeline and return a result row."""
    time, flux = load_light_curve(tic)                 # your Phase 1 loader
    time_c, flat, _ = detrend(time, flux)              # your detrend
    with contextlib.redirect_stdout(io.StringIO()):    # silence detect()'s prints
        r = detect(time_c, flat)                        # your TLS detection
    feats = extract_features(time_c, flat, r.period, r.T0, r.duration)
    feats["sde"] = float(r.SDE)
    feats["snr"] = float(r.snr)

    row = {"tic": tic, "period": float(r.period),
           "depth_ppm": (1 - r.depth) * 1e6,
           "duration_hr": float(r.duration) * 24,
           **feats, "status": "ok"}

    if clf is not None:
        Xr = pd.DataFrame([feats])[feature_cols]        # columns must match training
        row["predicted_class"] = clf.predict(Xr)[0]
        row["confidence"] = float(clf.predict_proba(Xr)[0].max())
    return row

In [42]:
def run_batch(tic_list, results_csv="results.csv", clf=None, feature_cols=None):
    """Process many stars into one CSV. Safe to stop and rerun -- it resumes."""
    done = set()
    if os.path.exists(results_csv):
        done = set(pd.read_csv(results_csv)["tic"])     # skip stars already finished
        print(f"Resuming: {len(done)} stars already done")

    for i, tic in enumerate(tic_list, 1):
        if tic in done:
            print(f"[{i}/{len(tic_list)}] {tic} already done, skipping")
            continue
        try:
            row = process_one_star(tic, clf, feature_cols)
        except Exception as e:                           # one bad star can't kill the run
            row = {"tic": tic, "status": f"error: {e}"}
        # append immediately so a crash never loses finished work
        pd.DataFrame([row]).to_csv(results_csv, mode="a",
                                   header=not os.path.exists(results_csv), index=False)
        print(f"[{i}/{len(tic_list)}] {tic} -> {row.get('predicted_class', row['status'])}")

    return pd.read_csv(results_csv)

In [ ]:
star_list = ["TIC 307210830", "TIC 231663901", "TIC 100100827"]

results = run_batch(star_list)    
results

Found 47 light curve files for TIC 307210830


100%|██████████| 2261/2261 periods | 00:23<00:00
100%|██████████| 7675/7675 [00:06<00:00, 1098.72it/s]


[1/3] TIC 307210830 -> ok
Found 12 light curve files for TIC 231663901


100%|██████████| 2392/2392 periods | 00:31<00:00


[2/3] TIC 231663901 -> ok
Found 14 light curve files for TIC 100100827


100%|██████████| 2351/2351 periods | 00:24<00:00


[3/3] TIC 100100827 -> ok


,tic,period,depth_ppm,duration_hr,primary_depth_ppm,secondary_depth_ppm,secondary_ratio,odd_even_diff,v_shape,sde,snr,status
0,TIC 307210830,3.691440,1716.694748,1.090888,1636.548635,-67.349160,-0.041153,0.008693,0.251816,20.920850,27.970756,ok
1,TIC 231663901,1.430880,17734.144600,1.310819,16969.905247,146.340359,0.008624,0.045784,0.326362,20.987298,106.386719,ok
2,TIC 100100827,0.941515,11371.744255,1.842160,10431.515476,414.472618,0.039733,0.002478,0.126963,42.637488,682.761308,ok
